# 07 · Photonic Quantum Circuits

**Covers:** a chip as one unitary matrix, the MZI as a tunable splitter, many photons and the matrix permanent, boson sampling, a three-mode "tritter", and quantum teleportation.
**Photonics thread:** programmable photonic chips are meshes of MZIs and phase shifters (Reck/Clements layouts). This notebook shows what such meshes do to several photons at once.

In [1]:
import numpy as np
import qutip as qt

results = []   # every theory-vs-simulation check is recorded here

def check(label, sim, theory, tol=1e-3):
    """Compare a simulated number with its analytic value and record the result."""
    sim, theory = float(np.real(sim)), float(np.real(theory))
    ok = bool(np.isclose(sim, theory, atol=tol))
    results.append(ok)
    print(f"{'PASS' if ok else 'FAIL'}  {label}: simulation = {sim:.4f}, theory = {theory:.4f}")

In [2]:
from math import factorial
from itertools import product

N = 3                                       # up to 2 photons per mode is enough here

def make_modes(m):
    ops = []
    for k in range(m):
        l = [qt.qeye(N)] * m
        l[k] = qt.destroy(N)
        ops.append(qt.tensor(l))
    return ops

def fock(pattern):
    return qt.tensor([qt.basis(N, k) for k in pattern])

def run(U, in_modes):
    # one photon into each listed input; rule: a_i^dagger -> sum_j U[j, i] a_j^dagger
    m = U.shape[0]
    a = make_modes(m)
    state = fock([0] * m)
    for i in in_modes:
        create = sum(complex(U[j, i]) * a[j].dag() for j in range(m))
        state = create * state
    return state

def prob(pattern, state):
    return abs(fock(pattern).dag() * state) ** 2

## 1. A chip is one unitary

**Plain words:** for $m$ waveguides the whole lossless chip is one $m\times m$ unitary $U$. A photon entering port $i$ leaves port $j$ with probability $|U_{ji}|^2$, and in general $a_i^\dagger \to \sum_j U_{ji} a_j^\dagger$.

The Mach–Zehnder from notebook 03 is a **tunable beam splitter**: $\tfrac12\begin{pmatrix}1+e^{i\varphi} & 1-e^{i\varphi}\\ 1-e^{i\varphi} & 1+e^{i\varphi}\end{pmatrix}$, and for two photons $P(1,1) = \cos^2\varphi$.

**Gap:** this holds for linear, lossless optics only. Loss breaks unitarity; photon–photon interaction needs nonlinearity or measurement tricks.

**One sentence:** any lossless chip is one unitary, built from a mesh of MZIs.

In [3]:
def mzi(phi):
    e = np.exp(1j * phi)
    return 0.5 * np.array([[1 + e, 1 - e], [1 - e, 1 + e]])

for phi in [0, np.pi / 4, np.pi / 2, np.pi]:
    check(f"MZI two-photon P(1,1) at phi = {phi:.3f}", prob((1, 1), run(mzi(phi), [0, 1])), np.cos(phi) ** 2)

PASS  MZI two-photon P(1,1) at phi = 0.000: simulation = 1.0000, theory = 1.0000
PASS  MZI two-photon P(1,1) at phi = 0.785: simulation = 0.5000, theory = 0.5000
PASS  MZI two-photon P(1,1) at phi = 1.571: simulation = 0.0000, theory = 0.0000
PASS  MZI two-photon P(1,1) at phi = 3.142: simulation = 1.0000, theory = 1.0000


## 2. Many photons: the permanent

**Plain words:** for several photons, add up every way they could travel through the chip. For one photon in inputs 0 and 1 and outputs $j, k$:

$$\text{amplitude} = U_{j0}U_{k1} + U_{j1}U_{k0} = \mathrm{perm}(U_{\text{sub}})$$

For the Hadamard the two terms are −½ and +½: the permanent is 0, which is the HOM dip. Subtracting instead gives the **determinant** (fermions), $|\det|^2 = 1$: fermions always separate. Bosons bunch, fermions don't, and the only difference is a sign.

**Boson sampling:** permanents of large matrices are believed to be very hard for classical computers, but a chip "computes" them by letting photons through. It is a hard sampling task used to show quantum advantage, not a useful algorithm.

**Gap:** loss and distinguishability push real boson sampling back toward classical behaviour.

**One sentence:** identical photons add their histories, and that sum (the permanent) is hard to compute classically.

In [4]:
Had = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
out = run(Had, [0, 1])
check("HOM from the general machinery: P(1,1)", prob((1, 1), out), 0.0)
check("fermions: |det|^2", abs(np.linalg.det(Had)) ** 2, 1.0)

w = np.exp(2j * np.pi / 3)                                   # three-mode Fourier "tritter"
F = np.array([[w ** (j * k) for k in range(3)] for j in range(3)]) / np.sqrt(3)
out3 = run(F, [0, 1])
check("norm (no truncation loss)", out3.norm(), 1.0)

def perm2(M):
    return M[0, 0] * M[1, 1] + M[0, 1] * M[1, 0]

coinc = 0.0
for pt in [(2, 0, 0), (0, 2, 0), (0, 0, 2), (1, 1, 0), (1, 0, 1), (0, 1, 1)]:
    modes = [j for j in range(3) for _ in range(pt[j])]
    theory = abs(perm2(F[np.ix_(modes, [0, 1])])) ** 2 / np.prod([factorial(k) for k in pt])
    sim = prob(pt, out3)
    check(f"tritter {pt}: |perm|^2 / n!", sim, theory)
    if max(pt) == 1:
        coinc += sim
distinguishable = 1 - sum(abs(F[j, 0]) ** 2 * abs(F[j, 1]) ** 2 for j in range(3))
check("coincidences, identical photons", coinc, 1 / 3)
check("coincidences, distinguishable photons", distinguishable, 2 / 3)

PASS  HOM from the general machinery: P(1,1): simulation = 0.0000, theory = 0.0000
PASS  fermions: |det|^2: simulation = 1.0000, theory = 1.0000
PASS  norm (no truncation loss): simulation = 1.0000, theory = 1.0000
PASS  tritter (2, 0, 0): |perm|^2 / n!: simulation = 0.2222, theory = 0.2222
PASS  tritter (0, 2, 0): |perm|^2 / n!: simulation = 0.2222, theory = 0.2222
PASS  tritter (0, 0, 2): |perm|^2 / n!: simulation = 0.2222, theory = 0.2222
PASS  tritter (1, 1, 0): |perm|^2 / n!: simulation = 0.1111, theory = 0.1111
PASS  tritter (1, 0, 1): |perm|^2 / n!: simulation = 0.1111, theory = 0.1111
PASS  tritter (0, 1, 1): |perm|^2 / n!: simulation = 0.1111, theory = 0.1111
PASS  coincidences, identical photons: simulation = 0.3333, theory = 0.3333
PASS  coincidences, distinguishable photons: simulation = 0.6667, theory = 0.6667


Identical photons give half as many coincidences as distinguishable ones: they bunch.

## 3. Teleportation: moving a state, not a photon

**Plain words:** Alice holds an unknown qubit and half of a Bell pair; Bob holds the other half. Alice makes a joint measurement on her two qubits (4 outcomes, ¼ each) and sends the 2-bit result by ordinary means. Bob applies a fix (X and/or Z) and his qubit becomes the original state, while Alice's is destroyed (no cloning).

**Photonics:** a linear-optics Bell measurement can only identify 2 of the 4 outcomes, so basic photonic teleportation succeeds half the time, and its core is HOM interference.

**Gap:** this simulation is the idealized logic, not the optical implementation. Nothing travels faster than light: Bob's qubit is useless until the 2 bits arrive.

**One sentence:** teleportation sends a state using an entangled pair plus 2 classical bits.

In [5]:
I2, X, Z = qt.qeye(2), qt.sigmax(), qt.sigmaz()
HadQ = qt.Qobj(Had)
P0, P1 = qt.ket2dm(qt.basis(2, 0)), qt.ket2dm(qt.basis(2, 1))

psi = (0.6 * qt.basis(2, 0) + 0.8j * qt.basis(2, 1)).unit()     # the unknown state
bell = (qt.tensor(qt.basis(2, 0), qt.basis(2, 0)) + qt.tensor(qt.basis(2, 1), qt.basis(2, 1))).unit()
CNOT01 = qt.tensor(P0, I2, I2) + qt.tensor(P1, X, I2)
state = qt.tensor(HadQ, I2, I2) * CNOT01 * qt.tensor(psi, bell)

for m0, m1 in product([0, 1], [0, 1]):
    proj = qt.tensor(qt.ket2dm(qt.basis(2, m0)), qt.ket2dm(qt.basis(2, m1)), I2)
    after = proj * state
    p = after.norm() ** 2
    bob = qt.ket2dm(after / after.norm()).ptrace(2)
    fix = (Z if m0 else I2) * (X if m1 else I2)
    raw = qt.expect(qt.ket2dm(psi), bob)
    fixed = qt.expect(qt.ket2dm(psi), fix * bob * fix.dag())
    print(f"outcome {(m0, m1)}: fidelity before fix = {raw:.3f}")
    check(f"outcome {(m0, m1)} probability", p, 0.25)
    check(f"outcome {(m0, m1)} fidelity after fix", fixed, 1.0)

outcome (0, 0): fidelity before fix = 1.000
PASS  outcome (0, 0) probability: simulation = 0.2500, theory = 0.2500
PASS  outcome (0, 0) fidelity after fix: simulation = 1.0000, theory = 1.0000
outcome (0, 1): fidelity before fix = 0.000
PASS  outcome (0, 1) probability: simulation = 0.2500, theory = 0.2500
PASS  outcome (0, 1) fidelity after fix: simulation = 1.0000, theory = 1.0000
outcome (1, 0): fidelity before fix = 0.078
PASS  outcome (1, 0) probability: simulation = 0.2500, theory = 0.2500
PASS  outcome (1, 0) fidelity after fix: simulation = 1.0000, theory = 1.0000
outcome (1, 1): fidelity before fix = 0.922
PASS  outcome (1, 1) probability: simulation = 0.2500, theory = 0.2500
PASS  outcome (1, 1) fidelity after fix: simulation = 1.0000, theory = 1.0000


## Validation summary

Every `check` above compared a QuTiP result with its analytic formula.

In [6]:
print(f"{sum(results)} of {len(results)} checks passed")
assert all(results), "a simulation disagrees with theory"

23 of 23 checks passed
